In [8]:
import json

from src.server.models.platform_values.models.platform_values_competitors_model import PlatformValuesCompetitorsCalc
from src.server.models.platform_values.models.platform_values_audience_angle_model import \
    PlatformValuesAudienceAngleCalc
from src.runtime_constants import rr
from src.server.models.obfuscator.int_id_obfuscator import UserObfuscator
from src.server.models.platform_values.models.platform_values_ts_discount_model import PlatformValuesTsDiscountModel
from src.server.models.platform_values.models.platform_values_product_nodes_audience_ts_model import PlatformValuesProductNodesAudienceTsModel
from src.db.mysql import DbConnection

app_id_int = 146 # clash of clans
dev_id_int = 32184 # supercell
platform_id = 1
survey_id = 3

obfuscator = UserObfuscator(multiplier=111577)

conn = DbConnection(rr.ENV_PROD, is_remote=True)

app_detail = conn.select_one("""
SELECT a.title, a.icon, p.tier
  FROM platform.platform_values_apps p,
       scraped_data.apps a
 WHERE a.app_id_int = %s
   AND p.app_id_int = a.app_id_int
""", [app_id_int])

tags_details = conn.select_all("""
SELECT t.tag_id_int, t.tag_rank
  FROM tagged_data.tags t
 WHERE t.app_id_int = %s
""", [app_id_int])

rows_product_nodes = conn.select_all("""
    SELECT p.tag_id_int,
           p.category,
           p.subcategory_int,
           m.subcategory,
           p.node,
           p.adj,
           p.is_survey,
           p.description,
           p.unlocked
      FROM app.def_sheet_platform_product p,
           app.map_tag_subcategory m
     WHERE p.is_prompt = 1
       AND m.id = p.subcategory_int
    """)

for row in rows_product_nodes:
    row["locked"] = 0
    if row["unlocked"] == 0:
        row["node"] = ""
        row["adj"] = ""
        row["description"] = ""
        row["locked"] = 1
    obfuscator.server_to_client_id(row)

rows_product_nodes.append({
    "tag_id": "__wip",
    "category": "Content",
    "subcategory_int": 0,
    "subcategory": "IPs",
    "node": "",
    "adj": "",
    "is_survey": 1,
    "description": "",
    "locked": 1,
    "wip": 1
})

rows_allowed_tags_per_locked_module = conn.select_all("""
    SELECT d.tag_id_int, d.module_id, d.is_set, d.is_loved, d.is_changeable
      FROM app.def_allowed_tags_per_locked_module d
    """)
for row in rows_allowed_tags_per_locked_module:
    obfuscator.server_to_client_id(row)

ds = PlatformValuesTsDiscountModel(
        conn=conn,
        platform_id=platform_id,
        dev_id_int=dev_id_int,
        survey_id=survey_id
    )

model = PlatformValuesProductNodesAudienceTsModel(
    conn=conn,
    platform_id=platform_id,
    survey_id=survey_id,
    dev_id_int=dev_id_int,
    ds=ds
)
# model.calc(
#     my_tier=app_detail["tier"],
#     selected_tags_details=tags_details
# )
product_nodes = model.generate_client_data(
    conn=conn,
    module_locked=True,
    platform_id=platform_id,
    obfuscator=obfuscator,
    survey_id=survey_id
)
model = PlatformValuesAudienceAngleCalc(
    conn=conn,
    platform_id=platform_id,
    survey_id=survey_id,
    ds=ds
)
# model.calc(
#     selected_ranked_tags_details=tags_details
# )
angles = model.generate_client_data(
    conn=conn,
    module_locked=True,
    platform_id=platform_id,
    obfuscator=obfuscator
)
model = PlatformValuesCompetitorsCalc(
    conn=conn,
    platform_id=platform_id,
    survey_id=survey_id
)
# model.find_competitors_for_audience_angle(
#     dev_id_int=dev_id_int,
#     my_tier=app_detail["tier"],
#     my_tags_details=tags_details,
#     audience_angle_platform_id=platform_id,
#     audience_angle_id=obfuscator.client_to_server(angles[0]["audience_angle_id"]),
#     exclude_apps_from_competitors=[]
# )
competitors = model.generate_client_data(
    conn=conn,
    module_locked=True,
    platform_id=platform_id,
    obfuscator=obfuscator
)
conn.close()

res = {
    "app_detail": app_detail,
    "tags_def": rows_product_nodes,
    "allowed_tags_per_locked_module": rows_allowed_tags_per_locked_module,
    "product_nodes": product_nodes,
    "angles": angles,
    "competitors": competitors,
    "selected_tags_details": tags_details
}

f = open("home_page_auditor_data.json", "a")
f.write(json.dumps(res))
f.close()

In [ ]:
conn.close()